In [25]:
import sys
sys.path.append("..")
from tools import hybrid_search, vector_search, fulltext_search
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from pathlib import Path
import json
from tqdm.auto import tqdm
import pandas as pd
from rag import RAG
from pydantic import BaseModel, Field
from typing import Literal

In [9]:
df_ground_truth = pd.read_csv("ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
Rag = RAG()

def generate_rag_answer(rec):
    question = rec["question"]
    answer_llm = Rag.rag(question)
    json_file = Path("samples") / rec["filename"]
    with json_file.open("r", encoding="utf-8") as f:
        data = json.load(f)
        content = data['content']

    return {
        "question": rec["question"],
        "title": rec["title"],
        "answer_llm": answer_llm,
        "filename": rec["filename"],
        "content": content,
    }



In [20]:
results = []
for rec in tqdm(ground_truth):
    results.append(generate_rag_answer(rec))


  0%|          | 0/55 [00:00<?, ?it/s]

In [21]:
df_answers = pd.DataFrame(results)
df_answers.to_csv("rag-answers.csv", index=False)

In [24]:
df_answers = pd.read_csv("rag-answers.csv")
answers = df_answers.to_dict(orient="records")

In [26]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [30]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a researcher or student
2. The content of the paper that is relevant to the question
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [31]:
aqa_judge_prompt = """
Question:
{question}

Content of the paper:
{content}

AI Answer:
{answer_llm}
""".strip()

In [34]:
def evaluate(question, content, answer_llm):
    prompt = aqa_judge_prompt.format(
        question=question,
        content=content,
        answer_llm=answer_llm
    )
    llm = ChatOpenAI(model="gpt-5.4-mini")
    structured_llm = llm.with_structured_output(AnswerEvaluation, include_raw=True)
    response = structured_llm.invoke(
        [
            ("system", aqa_judge_instructions),
            ("user", prompt),
        ]
    )
    answer = response["parsed"]

    return {
        "question": question,
        "answer_llm": answer_llm,
        "score": answer.score,
        "reasoning": answer.reasoning,
    } 

In [37]:
llm_judge_evals = []
for rec in tqdm(answers):
    eval_result = evaluate(
        question=rec["question"],
        content=rec["content"],
        answer_llm=rec["answer_llm"]
    )
    llm_judge_evals.append(eval_result)

  0%|          | 0/55 [00:00<?, ?it/s]

In [38]:
df_eval = pd.DataFrame(llm_judge_evals)

In [39]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 50/55 = 90.91%


In [40]:
df_eval[df_eval["score"] == "bad"].head()

,question,answer_llm,score,reasoning
13,How is PageRank actually computed for huge web...,PageRank on huge web graphs is usually compute...,bad,The answer captures the core idea that PageRan...
15,What are the main semantic models of functiona...,The main semantic models of functional reactiv...,bad,The answer correctly identifies Classic FRP an...
36,What is the syntax and idea behind the paralle...,"In CSP, the parallel construct is the `||` ope...",bad,The answer captures the main idea that CSP par...
39,Can you explain how CSP in the paper models th...,"In the CSP paper, these examples are modeled b...",bad,The answer captures the paper’s main idea that...
51,What were the main reasons the first-gen hardw...,The paper says first-generation hardware virtu...,bad,The answer captures the paper’s main point tha...


In [41]:
df_eval.to_csv("rag-evaluations.csv", index=False)